# Preference for Experimentation v2: Dynamic Programming

Julian Hsu

Initial version: 2026-06-16

## Motivation

The v1 notebook showed that a myopic agent — one that minimises a static loss function with fixed λ (variance aversion) and η (experimentation incentive) — cannot endogenously produce the right dynamics. An agent with high η over-experiments even after it already knows which arm is better; one with low η under-experiments and may never learn.

The fix is to solve the **infinite-horizon discounted dynamic programming problem** directly. The agent now accounts for the future value of information when choosing how much to experiment today.

## Model Setup

Each period $t$, the agent allocates proportion $p_t \in [0,1]$ to the treatment arm and $(1-p_t)$ to control. With $n$ users per round:

- $n_T = \text{round}(p_t \cdot n)$ observations go to treatment
- $n_C = n - n_T$ observations go to control

### Gaussian Normal-Normal Conjugate

The agent maintains Bayesian posteriors over each arm's mean:
$$\theta_i \sim \mathcal{N}(m_i, \, 1/\tau_i)$$

After $n_i$ cumulative observations, the posterior updates as:
$$\tau_i^{\text{new}} = \tau_i + n_i^{\text{new}} / \sigma^2, \qquad m_i^{\text{new}} = \frac{\tau_i m_i + (n_i^{\text{new}}/\sigma^2)\,\bar{y}_i^{\text{new}}}{\tau_i^{\text{new}}}$$

### State

The payoff each period is $p\,m_T + (1-p)\,m_C = m_C + p\,\Delta m$ where $\Delta m = m_T - m_C$. The argmax over $p$ depends only on $\Delta m$ (and the precisions), so we track the **state** $(\Delta m,\, n_T,\, n_C)$. The depth $d = (n_T + n_C)/n$ is implicit in the state; $n_C$ is no longer determined by a period index.

### Law of Motion

Precision evolves **deterministically**:
$$\tau_{i}^{\text{new}} = \tau_{i} + \Delta n_i / \sigma^2$$

The posterior mean difference evolves **stochastically** as a martingale:
$$\Delta m^{\text{new}} \mid \Delta m,\, p \;\sim\; \mathcal{N}\!\left(\Delta m,\; \underbrace{\frac{1}{\tau_T} - \frac{1}{\tau_T^{\text{new}}}}_{\text{treatment learning}} + \underbrace{\frac{1}{\tau_C} - \frac{1}{\tau_C^{\text{new}}}}_{\text{control learning}}\right)$$

The noise variance is exactly the total posterior variance reduced this period — zero when $p=0$ or $p=1$ for the non-allocated arm.

### Bellman Equation

The **infinite-horizon** Bellman equation for the stationary value function $V^*$:

$$V^*(\Delta m,\, n_T,\, n_C) = \max_{p \in P} \left\{ p\,\Delta m + \gamma\, \mathbb{E}\left[V^*(\Delta m',\, n_T + \Delta n_T,\, n_C + \Delta n_C)\right] \right\}$$

We solve via **fixed-point (value iteration)**: initialise $V^0 = 0$ and iterate
$$V^{k+1} = \mathcal{T}[V^k] \quad \text{until} \quad \|V^{k+1} - V^k\|_\infty < \varepsilon$$

The operator $\mathcal{T}$ is a contraction with modulus $\gamma$, so convergence is guaranteed. In practice, states with $n_T + n_C \ge N_{\max} \cdot n$ are treated as terminal ($V = 0$), which is accurate for $\gamma \le 0.9$ since $\gamma^{N_{\max}} \approx 0$.

The expectation over $\Delta m'$ is computed with **Gauss-Hermite quadrature** (10 nodes).

### Discount Factor

The sole free parameter is $\gamma \in [0,1)$:
- $\gamma = 0$: fully myopic — no value to learning; agent acts on current posterior only
- $\gamma \to 1$: patient agent; exploration has high option value

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermgauss

np.random.seed(42)

## DGP and Prior Parameters

Same data-generating process as v1. The agent never observes `true_effect` or `true_control` directly.

In [ ]:
# True DGP (never observed by agent)
true_effect  = -1.0   # treatment arm mean
true_control =  0.0   # control arm mean
sigma2       = 81.0   # observation noise variance (sigma=9, same as v1)
n_per_round  = 10     # users per period

# Agent's prior: N(m0, 1/tau0) for each arm
m0   = 0.0
tau0 = 1.0 / sigma2

# Coarse allocation grid: 5 values {0, 0.25, 0.5, 0.75, 1.0}
# With n_per_round=10, Python's banker's rounding gives dn_T ∈ {0, 2, 5, 8, 10}
# (0.25*10=2.5 → 2; 0.75*10=7.5 → 8), so actual splits are {0%, 20%, 50%, 80%, 100%}.
p_grid    = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
dn_T_grid = np.array([int(round(p * n_per_round)) for p in p_grid])
dn_C_grid = n_per_round - dn_T_grid

# DP parameters
gamma_values = [0.0, 0.5, 0.9]
T_dp  = 50    # value-iteration horizon (terminal boundary V[T_dp]=0)
T_sim = 20    # periods to simulate and plot
n_sim = 100   # Monte Carlo seeds for averaging

# State grid for Δm
dm_max  = 12.0
n_dm    = 121
dm_grid = np.linspace(-dm_max, dm_max, n_dm)

# Gauss-Hermite quadrature
n_quad      = 10
gh_xi, gh_w = hermgauss(n_quad)
gh_w       /= np.sqrt(np.pi)

print(f"Allocation grid  : p_grid  = {p_grid}")
print(f"Rounded dn_T     : {dn_T_grid}  (actual p = {dn_T_grid/n_per_round})")

In [ ]:
# ── Adaptive n_T bucketing ────────────────────────────────────────────────────
# Fine buckets when the posterior is still uncertain (low depth), coarse once
# the agent has enough data that precision gains are negligible.
#
#   depth d < 5  → bucket size 1  (exact counts; every integer tracked)
#   5 ≤ d < 15   → bucket size 5
#   d ≥ 15       → bucket size 20

def get_bucket_size(d):
    if d < 5:   return 1
    if d < 15:  return 5
    return 20

def make_nT_grid(d):
    """Sorted array of bucketed n_T values reachable at depth d."""
    B     = get_bucket_size(d)
    n_max = d * n_per_round
    grid  = np.arange(0, n_max + 1, B, dtype=int)
    if len(grid) == 0 or grid[-1] != n_max:   # always include the upper corner
        grid = np.append(grid, n_max)
    return grid

def snap_idx(n_T, grid):
    """Index of the grid element closest to n_T (grid is a sorted int array)."""
    idx = int(np.searchsorted(grid, n_T))
    if idx == 0:
        return 0
    if idx >= len(grid):
        return len(grid) - 1
    return int(idx - 1 if (n_T - grid[idx - 1]) <= (grid[idx] - n_T) else idx)

# Precompute grids for every depth including the terminal boundary d=T_dp
nT_grids = [make_nT_grid(d) for d in range(T_dp + 1)]

# ── State-space summary ───────────────────────────────────────────────────────
total_pairs_old = sum(d * n_per_round + 1 for d in range(T_dp))   # original B=1
total_pairs_new = sum(len(nT_grids[d])    for d in range(T_dp))   # adaptive

sweeps_old = total_pairs_old * 21       * n_quad   # old: 21 allocations
sweeps_new = total_pairs_new * len(p_grid) * n_quad   # new: 5 allocations

print(f"n_T state pairs  : {total_pairs_old:>6,}  →  {total_pairs_new:>4,}  ({total_pairs_old/total_pairs_new:.1f}× fewer)")
print(f"Allocations      :     21  →     5  (4.2× fewer)")
print(f"Bellman sweeps/iter: {sweeps_old:>8,}  →  {sweeps_new:>6,}  ({sweeps_old/sweeps_new:.1f}× fewer)")

## Fixed-Point Value Iteration

The state is $(\Delta m,\, n_T,\, n_C)$. Because each period allocates exactly $n$ observations, the depth $d = (n_T + n_C)/n$ is an integer; we index the value function by $d$, so $V[d]$ stores values over the $n_T$ grid at that depth.

### Adaptive $n_T$ bucketing

Tracking every exact integer $n_T$ is wasteful: the value function is nearly flat in $n_T$ once posteriors are tight. We use depth-dependent bucket sizes:

| Depth range | Bucket size $B$ | Rationale |
|---|---|---|
| $d < 5$ | 1 (exact) | Posterior still highly uncertain; small $n_T$ differences matter |
| $5 \le d < 15$ | 5 | Moderate data; 5-observation resolution sufficient |
| $d \ge 15$ | 20 | Posterior tight; coarse buckets lose almost nothing |

`make_nT_grid(d)` returns multiples of $B$ from 0 to $d \cdot n$, always including the endpoints 0 and $d \cdot n$. `snap_idx(n_T, grid)` maps any exact count to the nearest bucket index.

### Algorithm (Jacobi updates)

1. Initialise $V^0[d] = 0$ for all depths $d$.
2. For each iteration $k$ and depth $d$, for each $n_T$ in the bucketed grid:
$$V^{k+1}[d][\Delta m,\, j] = \max_{p}\!\left\{p\,\Delta m + \gamma\,\mathbb{E}_{\Delta m'}\!\left[V^k[d{+}1][\Delta m',\, \text{snap}(n_T{+}\Delta n_T,\, d{+}1)]\right]\right\}$$
3. Stop when $\|V^{k+1} - V^k\|_\infty < \varepsilon$.

Convergence requires $\approx \log(\varepsilon)/\log(\gamma)$ iterations.

In [ ]:
def solve_dp_fixedpoint(gamma, T_dp=T_dp, tol=1e-4, max_iter=500):
    """
    Fixed-point (value iteration) for the infinite-horizon discounted DP.

    State: (Δm, n_T, n_C), indexed by depth d = (n_T+n_C)//n_per_round.
    V[d] has shape (n_dm, len(nT_grids[d])).  V[T_dp] = 0 (terminal).

    n_T is tracked on the adaptive bucketed grid nT_grids[d]; next-state
    lookup uses snap_idx to map n_T+dn_T to the nearest bucket at d+1.

    Returns
    -------
    policy : list of length T_dp
        policy[d] has shape (n_dm, len(nT_grids[d]))
        policy[d][:, j] = optimal p for every Δm at the j-th n_T bucket
    """
    V = [np.zeros((n_dm, len(nT_grids[d]))) for d in range(T_dp + 1)]

    for iteration in range(max_iter):
        V_new = [None] * T_dp
        delta  = 0.0

        for d in range(T_dp):
            grid_d   = nT_grids[d]
            grid_dp1 = nT_grids[d + 1]
            n_nT     = len(grid_d)
            V_d_new  = np.full((n_dm, n_nT), -np.inf)

            for j, n_T in enumerate(grid_d):
                n_C   = d * n_per_round - n_T
                tau_T = tau0 + n_T / sigma2
                tau_C = tau0 + n_C / sigma2

                for dn_T, dn_C in zip(dn_T_grid, dn_C_grid):
                    p_actual   = dn_T / n_per_round
                    cur_payoff = p_actual * dm_grid

                    if gamma == 0.0:
                        val = cur_payoff
                    else:
                        tau_T_new = tau_T + dn_T / sigma2
                        tau_C_new = tau_C + dn_C / sigma2

                        var_dm = 0.0
                        if dn_T > 0:
                            var_dm += 1.0/tau_T - 1.0/tau_T_new
                        if dn_C > 0:
                            var_dm += 1.0/tau_C - 1.0/tau_C_new
                        std_dm = np.sqrt(max(var_dm, 0.0))

                        j_next = snap_idx(n_T + dn_T, grid_dp1)
                        v_col  = V[d + 1][:, j_next]

                        exp_future = np.zeros(n_dm)
                        for q in range(n_quad):
                            dm_next     = dm_grid + np.sqrt(2.0) * std_dm * gh_xi[q]
                            exp_future += gh_w[q] * np.interp(dm_next, dm_grid, v_col)

                        val = cur_payoff + gamma * exp_future

                    better        = val > V_d_new[:, j]
                    V_d_new[:, j] = np.where(better, val, V_d_new[:, j])

            delta    = max(delta, np.max(np.abs(V_d_new - V[d])))
            V_new[d] = V_d_new

        for d in range(T_dp):
            V[d] = V_new[d]

        if (iteration + 1) % 10 == 0 or delta < tol:
            print(f"    iter {iteration+1:4d}  ||ΔV||_inf = {delta:.3e}")
        if delta < tol:
            print(f"  Converged.")
            break
    else:
        print(f"  Did not converge after {max_iter} iterations (delta={delta:.3e})")

    # Extract stationary policy from converged V*
    policy = [None] * T_dp
    for d in range(T_dp):
        grid_d   = nT_grids[d]
        grid_dp1 = nT_grids[d + 1]
        n_nT     = len(grid_d)
        pol_d    = np.zeros((n_dm, n_nT))

        for j, n_T in enumerate(grid_d):
            n_C   = d * n_per_round - n_T
            tau_T = tau0 + n_T / sigma2
            tau_C = tau0 + n_C / sigma2
            best_val = np.full(n_dm, -np.inf)
            best_p   = np.zeros(n_dm)

            for dn_T, dn_C in zip(dn_T_grid, dn_C_grid):
                p_actual   = dn_T / n_per_round
                cur_payoff = p_actual * dm_grid

                if gamma == 0.0:
                    val = cur_payoff
                else:
                    tau_T_new = tau_T + dn_T / sigma2
                    tau_C_new = tau_C + dn_C / sigma2

                    var_dm = 0.0
                    if dn_T > 0:
                        var_dm += 1.0/tau_T - 1.0/tau_T_new
                    if dn_C > 0:
                        var_dm += 1.0/tau_C - 1.0/tau_C_new
                    std_dm = np.sqrt(max(var_dm, 0.0))

                    j_next = snap_idx(n_T + dn_T, grid_dp1)
                    v_col  = V[d + 1][:, j_next]

                    exp_future = np.zeros(n_dm)
                    for q in range(n_quad):
                        dm_next     = dm_grid + np.sqrt(2.0) * std_dm * gh_xi[q]
                        exp_future += gh_w[q] * np.interp(dm_next, dm_grid, v_col)

                    val = cur_payoff + gamma * exp_future

                better   = val > best_val
                best_val = np.where(better, val,      best_val)
                best_p   = np.where(better, p_actual, best_p)

            pol_d[:, j] = best_p

        policy[d] = pol_d

    return policy


print("Solving fixed-point Bellman for each gamma...")
policies = {}
for g in gamma_values:
    print(f"\n  gamma={g}")
    policies[g] = solve_dp_fixedpoint(g)
print("\nDone.")

## Simulation

For each $\gamma$, we simulate `n_sim` independent runs of `T_sim` periods using the **stationary** policy $\pi^*(\Delta m, n_T, n_C)$.

At each period the agent:
1. Computes the state $(\Delta m_t,\, n_{T,t},\, n_{C,t})$ and the depth $d_t = (n_{T,t} + n_{C,t})/n$
2. Looks up $p^* = \pi^*[d_t][\Delta m_t,\, n_{T,t}]$ (with linear interpolation over $\Delta m$)
3. Draws outcomes from the true DGP
4. Updates posteriors via Bayesian updating
5. Records allocation and cumulative payoff

Because the policy is stationary, the same $\pi^*$ is used at every period — the dynamics emerge purely from how the state evolves.

In [ ]:
def simulate(policy, seed=0):
    rng = np.random.default_rng(seed)

    m_T, tau_T = m0, tau0
    m_C, tau_C = m0, tau0
    n_T_cum = 0
    n_C_cum = 0

    alloc_hist   = []
    cumrew_hist  = []
    total_reward = 0.0

    actual_p_vals = np.unique(dn_T_grid / n_per_round)

    for _ in range(T_sim):
        dm    = m_T - m_C
        d     = (n_T_cum + n_C_cum) // n_per_round
        pol_d = policy[d]

        # Snap exact n_T_cum to the nearest bucket index for this depth
        j      = snap_idx(n_T_cum, nT_grids[d])
        p_star = np.interp(dm, dm_grid, pol_d[:, j])
        p_star = actual_p_vals[np.argmin(np.abs(actual_p_vals - p_star))]

        dn_T = int(round(p_star * n_per_round))
        dn_C = n_per_round - dn_T

        y_T = rng.normal(true_effect,  np.sqrt(sigma2), dn_T) if dn_T > 0 else np.array([])
        y_C = rng.normal(true_control, np.sqrt(sigma2), dn_C) if dn_C > 0 else np.array([])

        total_reward += y_T.sum() + y_C.sum()
        alloc_hist.append(p_star)
        cumrew_hist.append(total_reward)

        if dn_T > 0:
            tau_T_new = tau_T + dn_T / sigma2
            m_T       = (tau_T * m_T + (dn_T / sigma2) * y_T.mean()) / tau_T_new
            tau_T     = tau_T_new
        if dn_C > 0:
            tau_C_new = tau_C + dn_C / sigma2
            m_C       = (tau_C * m_C + (dn_C / sigma2) * y_C.mean()) / tau_C_new
            tau_C     = tau_C_new

        n_T_cum += dn_T
        n_C_cum += dn_C

    return np.array(alloc_hist), np.array(cumrew_hist)


results = {}
for g in gamma_values:
    allocs, cumrews = [], []
    for s in range(n_sim):
        a, r = simulate(policies[g], seed=s)
        allocs.append(a)
        cumrews.append(r)
    results[g] = {
        'alloc':  np.array(allocs),
        'cumrew': np.array(cumrews),
    }
    print(f"gamma={g}: mean allocation={np.array(allocs).mean():.3f}")

In [ ]:
# ── Unit tests ────────────────────────────────────────────────────────────────
def run_tests():
    failures = []

    def check(cond, msg):
        if not cond:
            failures.append(msg)

    # 1. Bucket size thresholds
    check(get_bucket_size(0)  == 1,  "bucket_size d=0  should be 1")
    check(get_bucket_size(4)  == 1,  "bucket_size d=4  should be 1")
    check(get_bucket_size(5)  == 5,  "bucket_size d=5  should be 5")
    check(get_bucket_size(14) == 5,  "bucket_size d=14 should be 5")
    check(get_bucket_size(15) == 20, "bucket_size d=15 should be 20")
    check(get_bucket_size(49) == 20, "bucket_size d=49 should be 20")

    # 2. nT grids: start at 0, end at d*n_per_round, strictly increasing
    for d in [0, 1, 4, 5, 14, 15, 30, 49]:
        g     = nT_grids[d]
        n_max = d * n_per_round
        check(int(g[0]) == 0,     f"nT_grid[{d}] must start at 0, got {g[0]}")
        check(int(g[-1]) == n_max, f"nT_grid[{d}] must end at {n_max}, got {g[-1]}")
        check(np.all(np.diff(g) > 0), f"nT_grid[{d}] must be strictly increasing")
        check(len(g) >= 1,         f"nT_grid[{d}] must be non-empty")

    # 3. snap_idx always returns a valid index
    for d in [0, 5, 15, 49]:
        g = nT_grids[d]
        for n_T in [0, 1, d * n_per_round // 2, d * n_per_round]:
            idx = snap_idx(n_T, g)
            check(0 <= idx < len(g),
                  f"snap_idx({n_T}, d={d}) returned {idx}, out of [0, {len(g)})")

    # 4. State space is substantially smaller than the original dense grid
    total_pairs = sum(len(nT_grids[d]) for d in range(T_dp))
    original    = sum(d * n_per_round + 1 for d in range(T_dp))
    check(total_pairs < original / 5,
          f"state pairs {total_pairs} should be <1/5 of original {original}")
    check(total_pairs > 200,
          f"state pairs {total_pairs} suspiciously small (sanity floor 200)")

    # 5. Policy shapes and value ranges
    for g in gamma_values:
        pol = policies[g]
        check(len(pol) == T_dp, f"gamma={g}: policy length should be {T_dp}")
        for d in [0, 1, 5, 15, T_dp - 1]:
            expected = (n_dm, len(nT_grids[d]))
            check(pol[d].shape == expected,
                  f"gamma={g}, d={d}: shape {pol[d].shape} != {expected}")
            check(np.all(pol[d] >= -1e-9) and np.all(pol[d] <= 1 + 1e-9),
                  f"gamma={g}, d={d}: policy values outside [0,1]")
            check(np.all(np.isfinite(pol[d])),
                  f"gamma={g}, d={d}: policy contains non-finite values")

    # 6. Gamma=0 is purely greedy at d=0 (single bucket, n_T=0, n_C=0)
    #    When Δm < 0 (treatment bad), p* = 0; when Δm > 0 (treatment good), p* = 1.
    pol0_d0 = policies[0.0][0][:, 0]   # shape (n_dm,)
    neg_mask = dm_grid < -2.0
    pos_mask = dm_grid >  2.0
    check(np.all(pol0_d0[neg_mask] == 0.0),
          "gamma=0, d=0: p should be 0 when Δm < -2 (greedy control)")
    check(np.all(pol0_d0[pos_mask] == 1.0),
          "gamma=0, d=0: p should be 1 when Δm > +2 (greedy treatment)")

    # 7. Patient agent explores more than myopic in early periods
    #    gamma=0.9 should have mean allocation closer to 0.5 over first 5 periods.
    early_alloc_09 = results[0.9]['alloc'][:, :5].mean()
    early_alloc_00 = results[0.0]['alloc'][:, :5].mean()
    check(abs(early_alloc_09 - 0.5) < abs(early_alloc_00 - 0.5),
          f"gamma=0.9 early mean {early_alloc_09:.3f} should be closer to 0.5 "
          f"than gamma=0 mean {early_alloc_00:.3f}")

    # 8. Simulate produces finite arrays of the right length
    for g in gamma_values:
        a, r = simulate(policies[g], seed=9999)
        check(a.shape == (T_sim,),           f"gamma={g}: alloc shape {a.shape} != ({T_sim},)")
        check(np.all((a >= 0) & (a <= 1)),   f"gamma={g}: allocations outside [0,1]")
        check(np.all(np.isfinite(r)),         f"gamma={g}: cumrew has non-finite values")
        check(r.shape == (T_sim,),            f"gamma={g}: cumrew shape {r.shape} != ({T_sim},)")

    # ── Report ────────────────────────────────────────────────────────────────
    n_tests = 6 + 3*len(gamma_values) + 4*8 + 2*len(gamma_values)  # rough count
    if failures:
        print(f"FAILED  ({len(failures)} assertion(s)):")
        for f in failures:
            print(f"  ✗  {f}")
    else:
        print(f"All tests passed  (state pairs: {total_pairs:,} vs original {original:,},"
              f"  {original/total_pairs:.1f}× reduction)")

run_tests()

## Results

In [ ]:
periods = np.arange(1, T_sim + 1)
colors  = {0.0: '#e41a1c', 0.5: '#377eb8', 0.9: '#4daf4a'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Treatment Allocation ──────────────────────────────────────────────
ax = axes[0]
for g in gamma_values:
    m = results[g]['alloc'].mean(axis=0)
    s = results[g]['alloc'].std(axis=0)
    ax.plot(periods, m, color=colors[g], label=f'γ = {g}')
    ax.fill_between(periods, m - s, m + s, color=colors[g], alpha=0.12)

ax.axhline(0.5, linestyle='--', color='gray', linewidth=0.8, label='Balanced (0.5)')
ax.axhline(0.0, linestyle=':',  color='black', linewidth=0.8, label='Optimal (p=0)')
ax.set_title('Treatment Allocation Over Time\n(mean ± 1 SD across simulations)')
ax.set_xlabel('Period')
ax.set_ylabel('Proportion Treated  (p)')
ax.set_ylim(-0.05, 1.05)
ax.legend()

# ── Plot 2: Cumulative Payoff ─────────────────────────────────────────────────
ax = axes[1]

# Benchmarks
optimal_cumrew = np.zeros(T_sim)                          # all control: E[payoff]=0
worst_cumrew   = true_effect * n_per_round * periods      # all treatment
ax.plot(periods, optimal_cumrew, linestyle='--', color='black',  linewidth=1.0, label='Optimal (p=0)')
ax.plot(periods, worst_cumrew,   linestyle=':',  color='dimgray', linewidth=1.0, label='Worst (p=1)')

for g in gamma_values:
    m = results[g]['cumrew'].mean(axis=0)
    s = results[g]['cumrew'].std(axis=0)
    ax.plot(periods, m, color=colors[g], label=f'γ = {g}')
    ax.fill_between(periods, m - s, m + s, color=colors[g], alpha=0.12)

ax.set_title('Cumulative Payoff Over Time\n(mean ± 1 SD across simulations)')
ax.set_xlabel('Period')
ax.set_ylabel('Cumulative Payoff')
ax.legend()

plt.tight_layout()
plt.savefig('preference_experimentation_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## Takeaway

The plots show the first `T_sim` periods of behaviour under the **stationary** infinite-horizon policy. The policy is computed once via fixed-point value iteration and is a function of the state $(\Delta m, n_T, n_C)$ alone — there is no explicit period index.

- **γ = 0 (myopic):** No value to future learning. The agent always plays the arm with the higher current posterior mean — pure greedy exploitation. Early allocations are noisy and heavily influenced by the prior.

- **γ = 0.5 (moderate):** Some option value to learning. The agent experiments more than the myopic case early on but converges to the correct arm faster in terms of payoff, because it does not invest heavily in learning an arm it already suspects is inferior.

- **γ = 0.9 (patient):** High option value to information. The agent allocates substantially to both arms early on to reduce posterior uncertainty. This costs cumulative payoff in the short run but leads to more confident convergence to the optimal policy (all control). The exploration incentive decays endogenously: as precisions grow, $\sigma^2_{\Delta m} \to 0$, the Bellman term adds no exploration bonus, and the policy naturally collapses to pure exploitation.

**Fixed-point vs backward induction.** Backward induction (single backward sweep from the terminal condition) is equivalent to running value iteration once with a Gauss-Seidel update order — it gives the exact $V^*$ for the finite-horizon problem in one pass. The fixed-point (Jacobi) approach used here iterates the Bellman operator globally until convergence, giving the genuine infinite-horizon $V^*$. The policies agree when the terminal horizon is large relative to $1/(1-\gamma)$, but the fixed-point formulation is conceptually cleaner: the policy is stationary, and there is no artificial terminal date in the agent's problem.